In [1]:
import json
from collections import Counter
import pandas as pd

In [5]:
with open("test_results/biomedbert_test_result.json", "r", encoding="utf-8") as f:
    biomedbert_corpus = json.load(f)
print(biomedbert_corpus[0])

{'sample_index': 0, 'text_stream': [{'token': 'non', 'true': ['T131'], 'pred': ['T109']}, {'token': 'diet', 'true': ['T131'], 'pred': ['T109']}, {'token': 'inhibits', 'true': ['T052'], 'pred': ['T052']}, {'token': 'apoptosis', 'true': ['T043'], 'pred': ['T043']}, {'token': 'induced', 'true': ['T169'], 'pred': ['T169']}, {'token': 'in', 'true': ['O'], 'pred': ['O']}, {'token': 'pc12', 'true': ['T025'], 'pred': ['T025']}, {'token': 'cells', 'true': ['T025'], 'pred': ['T025']}, {'token': 'non', 'true': ['T131'], 'pred': ['T109']}, {'token': 'and', 'true': ['O'], 'pred': ['O']}, {'token': 'short', 'true': ['T131'], 'pred': ['T109']}, {'token': 'chain', 'true': ['T131'], 'pred': ['T109']}, {'token': 'non', 'true': ['T131'], 'pred': ['T109']}, {'token': 'eth', 'true': ['T131'], 'pred': ['T109']}, {'token': 'such', 'true': ['O'], 'pred': ['O']}, {'token': 'as', 'true': ['O'], 'pred': ['O']}, {'token': 'np', 'true': ['T131'], 'pred': ['T109']}, {'token': 'eo', 'true': ['T131'], 'pred': ['T109'

In [ ]:
def analyze_exact_tag_combination(corpus, target_tags):
    """
    Filters the corpus for tokens where the true labels are EXACTLY 
    the combination specified in 'target_tags' (order-independent).
    
    Parameters:
    - corpus: The qualitative_corpus list of dictionaries.
    - target_tags: A list of strings, e.g., ["T131"] or ["T131", "T024"]
    """
    # Ensure target_tags is a list and sort it for reliable matching
    if isinstance(target_tags, str):
        target_tags = [target_tags]
    target_tags_sorted = sorted(target_tags)
    target_tags_str = ", ".join(target_tags_sorted)
    
    prediction_counts = Counter()
    matched_examples = []
    
    # Scan through the corpus
    for sample in corpus:
        for token_info in sample["text_stream"]:
            current_true_sorted = sorted(token_info["true"])
            
            if current_true_sorted == target_tags_sorted:
                pred_combo = ", ".join(sorted(token_info["pred"])) if token_info["pred"] else "O"
                prediction_counts[pred_combo] += 1
                
                matched_examples.append({
                    "token": token_info["token"],
                    "gold tag": target_tags,
                    "predicted": pred_combo,
                    "sample_idx": sample["sample_index"]
                })
                
    total_matches = sum(prediction_counts.values())
    
    if total_matches == 0:
        print(f"Empty for tag: [{target_tags_str}]")
        return None, []
        
  
    df_summary = pd.DataFrame(prediction_counts.items(), columns=["Predicted Combination", "Count"])
    df_summary = df_summary.sort_values(by="Count", ascending=False).reset_index(drop=True)
    df_summary["Percentage"] = ((df_summary["Count"] / total_matches) * 100).round(2).astype(str) + "%"
    
    print(f"Gold Target: [{target_tags_str}] (Total Occurrences: {total_matches})")
    return df_summary, matched_examples

In [13]:
analyze_exact_tag_combination(biomedbert_corpus, "T131")

--- Analysis for Gold Target: [T131] (Total Occurrences: 159) ---


(  Predicted Combination  Count Percentage
 0                     O     87     54.72%
 1                  T109     43     27.04%
 2                  T131      8      5.03%
 3            T109, T131      7       4.4%
 4                  T196      5      3.14%
 5                  T121      4      2.52%
 6            T109, T121      2      1.26%
 7                  T073      2      1.26%
 8                  T080      1      0.63%,
 [{'token': 'non', 'gold tag': ['T131'], 'predicted': 'T109', 'sample_idx': 0},
  {'token': 'diet',
   'gold tag': ['T131'],
   'predicted': 'T109',
   'sample_idx': 0},
  {'token': 'non', 'gold tag': ['T131'], 'predicted': 'T109', 'sample_idx': 0},
  {'token': 'short',
   'gold tag': ['T131'],
   'predicted': 'T109',
   'sample_idx': 0},
  {'token': 'chain',
   'gold tag': ['T131'],
   'predicted': 'T109',
   'sample_idx': 0},
  {'token': 'non', 'gold tag': ['T131'], 'predicted': 'T109', 'sample_idx': 0},
  {'token': 'eth', 'gold tag': ['T131'], 'predicted': 'T1